# Wildfire Canada — Full Model Walkthrough (v3 → v14.1)

This notebook walks through the model's real development history, version by version, using the actual working code from each stage — with dead ends, crashes, and abandoned retries removed. It mirrors `CHANGELOG.md` in the repo root.

It won't run top-to-bottom without your own data (NFDB database, Earth Engine auth, FIRMS API key) — it's a readable record of the actual pipeline, not a packaged reproducible script. For a runnable version, see `data_pipeline/` and `modeling/`.


## Setup

Load the cleaned fire database (built once, reused across every version below).

In [ ]:
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount("/content/drive")

import shutil
LOCAL_DB = "/content/wildfire_corrected.db"
DRIVE_DB = "/content/drive/MyDrive/wildfire_project/wildfire_corrected.db"
shutil.copy(DRIVE_DB, LOCAL_DB)

import sqlite3
conn = sqlite3.connect(LOCAL_DB)
df_nfdb_clean = pd.read_sql("SELECT * FROM fires_v4_no_duplicates", conn)
weather = pd.read_sql("SELECT * FROM weather_daily", conn)
env_df = pd.read_sql("SELECT * FROM environment_data", conn)
modis_agg_fixed = pd.read_sql("SELECT * FROM modis_agg_fixed_early", conn)
viirs_agg_fixed = pd.read_sql("SELECT * FROM viirs_agg_fixed_early", conn)
conn.close()

df_nfdb_clean["REP_DATE"] = pd.to_datetime(df_nfdb_clean["REP_DATE"], errors="coerce")
df_nfdb_2012plus = df_nfdb_clean[df_nfdb_clean.REP_DATE.dt.year >= 2012].copy()
weather["date"] = pd.to_datetime(weather["date"])

# Leakage fix: only weather from BEFORE the report date
weather_no_leak = weather.merge(
    df_nfdb_2012plus[["NFDBFIREID", "REP_DATE"]], left_on="fire_id", right_on="NFDBFIREID", how="left"
)
weather_no_leak = weather_no_leak[weather_no_leak["date"] < weather_no_leak["REP_DATE"]]
weather_sorted = weather_no_leak.sort_values(["fire_id", "date"])

weather_7d = weather_sorted.groupby("fire_id").tail(7).groupby("fire_id").agg({
    "temperature_2m_max": ["mean", "max", "min"], "temperature_2m_min": "mean",
    "precipitation_sum": ["sum", "mean"], "wind_speed_10m_max": ["mean", "max"],
    "wind_gusts_10m_max": "mean", "relative_humidity_2m_mean": ["mean", "min", "max"],
    "sunshine_duration": "mean",
}).reset_index()
weather_7d.columns = ["_".join(col).strip("_") for col in weather_7d.columns.values]
weather_7d.rename(columns={"fire_id": "NFDBFIREID"}, inplace=True)

env_summary = env_df.groupby("fire_id").first().reset_index()
env_summary.rename(columns={"fire_id": "NFDBFIREID"}, inplace=True)

print(f"Fires loaded: {len(df_nfdb_clean):,}")
print(f"Fires with 7-day weather: {len(weather_7d):,}")


## v3 / v4 — first models

Random Forest baseline. v4 added two extra features (`dry_days_streak_14d`, `slope_aspect_interaction`) as an experiment — both were dropped again in v6 after not earning their place.

In [ ]:
df_model = df_nfdb_2012plus[["NFDBFIREID", "SIZE_HA", "CAUSE", "SRC_AGENCY", "REP_DATE", "YEAR"]].merge(
    weather_7d, on="NFDBFIREID", how="inner"
)
df_model["month"] = df_model["REP_DATE"].dt.month
df_model["is_big_fire"] = (df_model["SIZE_HA"] > 100).astype(int)
df_model = df_model.merge(env_summary[["NFDBFIREID", "NDVI", "elevation", "slope"]], on="NFDBFIREID", how="left")

weather_env_cols = [c for c in df_model.columns if "temperature" in c or "precipitation" in c
                    or "wind" in c or "humidity" in c or "sunshine" in c]

df_model_v3 = df_model.merge(modis_agg_fixed, on="NFDBFIREID", how="left").merge(viirs_agg_fixed, on="NFDBFIREID", how="left")
satellite_cols_fixed = ["modis_count_early", "modis_max_frp_early", "viirs_count_early", "viirs_max_frp_early"]
for col in satellite_cols_fixed:
    df_model_v3[col] = df_model_v3[col].fillna(0)

feature_cols_v3 = weather_env_cols + ["NDVI", "elevation", "slope"] + satellite_cols_fixed
print("df_model_v3:", df_model_v3.shape)


In [ ]:
# v4 experiment: dry-day streaks + slope/aspect interaction (later dropped)
def count_dry_days(group, threshold=0.5):
    group = group.sort_values("date")
    dry_streak, max_streak = 0, 0
    for precip in group["precipitation_sum"].values:
        if precip < threshold:
            dry_streak += 1
            max_streak = max(max_streak, dry_streak)
        else:
            dry_streak = 0
    return max_streak

dry_days_features = []
for fire_id, group in weather_no_leak.groupby("fire_id"):
    last14 = group.sort_values("date").tail(14)
    dry_days_features.append({"NFDBFIREID": fire_id, "dry_days_streak_14d": count_dry_days(last14)})
dry_days_df = pd.DataFrame(dry_days_features)

env_summary["slope_aspect_interaction"] = env_summary["slope"] * np.cos(np.radians(env_summary["aspect"]))

df_model_v4 = df_model_v3.merge(dry_days_df, on="NFDBFIREID", how="left")
df_model_v4 = df_model_v4.merge(env_summary[["NFDBFIREID", "slope_aspect_interaction"]], on="NFDBFIREID", how="left")
df_model_v4["dry_days_streak_14d"] = df_model_v4["dry_days_streak_14d"].fillna(0)
df_model_v4["slope_aspect_interaction"] = df_model_v4["slope_aspect_interaction"].fillna(
    df_model_v4["slope_aspect_interaction"].mean())

feature_cols_v4 = feature_cols_v3 + ["dry_days_streak_14d", "slope_aspect_interaction"]
X_v4 = df_model_v4[feature_cols_v4].copy()
y_v4 = df_model_v4["is_big_fire"]

X_train, X_test, y_train, y_test = train_test_split(X_v4, y_v4, test_size=0.2, random_state=42, stratify=y_v4)
X_train_bal, y_train_bal = RandomOverSampler(random_state=42).fit_resample(X_train, y_train)

rf_model_v4 = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf_model_v4.fit(X_train_bal, y_train_bal)
joblib.dump(rf_model_v4, "model_v4_rf.pkl")


## v6 — correctly-bounded satellite features

Fixed the satellite pre-detection window so only detections in the 7 days *before* the official report count (an earlier version let post-report detections leak in). Dropped v4's dry-day/aspect features — they didn't carry their weight.

In [ ]:
df_model_fixed = df_nfdb_2012plus[["NFDBFIREID", "SIZE_HA", "CAUSE", "SRC_AGENCY", "REP_DATE", "YEAR"]].merge(
    weather_7d, on="NFDBFIREID", how="inner"
)
df_model_fixed["month"] = df_model_fixed["REP_DATE"].dt.month
df_model_fixed["is_big_fire"] = (df_model_fixed["SIZE_HA"] > 100).astype(int)
df_model_fixed = df_model_fixed.merge(env_summary[["NFDBFIREID", "NDVI", "elevation", "slope"]], on="NFDBFIREID", how="left")
df_model_fixed = df_model_fixed.merge(modis_agg_fixed, on="NFDBFIREID", how="left")
df_model_fixed = df_model_fixed.merge(viirs_agg_fixed, on="NFDBFIREID", how="left")

sat_cols_new = ["modis_count_early7d", "modis_max_frp_early7d", "viirs_count_early7d", "viirs_max_frp_early7d"]
for col in sat_cols_new:
    df_model_fixed[col] = df_model_fixed[col].fillna(0)

feature_cols_fixed = weather_env_cols + ["NDVI", "elevation", "slope"] + sat_cols_new
X_fixed = df_model_fixed[feature_cols_fixed].copy()
for col in ["NDVI", "elevation", "slope"]:
    X_fixed[col] = X_fixed[col].fillna(X_fixed[col].mean())
y_fixed = df_model_fixed["is_big_fire"]

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(X_fixed, y_fixed, test_size=0.2, random_state=42, stratify=y_fixed)
X_train_f_bal, y_train_f_bal = RandomOverSampler(random_state=42).fit_resample(X_train_f, y_train_f)

model_fixed = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
model_fixed.fit(X_train_f_bal, y_train_f_bal)

proba_f = model_fixed.predict_proba(X_test_f)[:, 1]
print(f"ROC-AUC: {roc_auc_score(y_test_f, proba_f):.3f} | PR-AUC: {average_precision_score(y_test_f, proba_f):.3f}")

joblib.dump({"model": model_fixed, "feature_cols": feature_cols_fixed}, "model_v6_bounded_satellite_BEST.pkl")


## v7 — added province

Adding province changed elevation's feature importance meaningfully — a sign some of what "elevation" captured was really regional variation.

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_model_prov = df_model_fixed.merge(
    df_nfdb_2012plus[["NFDBFIREID", "SRC_AGENCY"]], on="NFDBFIREID", how="left", suffixes=("", "_dup")
)
le_prov = LabelEncoder()
df_model_prov["province_encoded"] = le_prov.fit_transform(df_model_prov["SRC_AGENCY"].astype(str))

feature_cols_prov = feature_cols_fixed + ["province_encoded"]
X_prov = df_model_prov[feature_cols_prov].copy()
for col in ["NDVI", "elevation", "slope"]:
    X_prov[col] = X_prov[col].fillna(X_prov[col].mean())
y_prov = df_model_prov["is_big_fire"]

X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(X_prov, y_prov, test_size=0.2, random_state=42, stratify=y_prov)
X_train_p_bal, y_train_p_bal = RandomOverSampler(random_state=42).fit_resample(X_train_p, y_train_p)

model_prov = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
model_prov.fit(X_train_p_bal, y_train_p_bal)

proba_p = model_prov.predict_proba(X_test_p)[:, 1]
print(f"ROC-AUC: {roc_auc_score(y_test_p, proba_p):.3f} | PR-AUC: {average_precision_score(y_test_p, proba_p):.3f}")

importance = pd.Series(model_prov.feature_importances_, index=feature_cols_prov).sort_values(ascending=False)
print(importance)

joblib.dump({"model": model_prov, "feature_cols": feature_cols_prov, "label_encoder_province": le_prov},
            "model_v7_with_province_BEST.pkl")


## v10 — switched to LightGBM + temporal split

Replaced Random Forest with LightGBM, and switched from a random train/test split to a **temporal** split — train on earlier years, test on later years. This is the first step toward the forward-testing discipline used for every version after this.

In [ ]:
feature_cols_v10 = weather_env_cols + ["NDVI", "elevation", "slope", "province_encoded"] + sat_cols_new
df_model_v10 = df_model_prov.copy()  # same features/frame as v7, temporal split instead of random

train_t = df_model_v10[df_model_v10.YEAR <= 2021]
test_t = df_model_v10[df_model_v10.YEAR > 2021]

X_train_t10 = train_t[feature_cols_v10].fillna(train_t[feature_cols_v10].mean())
X_test_t10 = test_t[feature_cols_v10].fillna(train_t[feature_cols_v10].mean())
y_train_t10, y_test_t10 = train_t["is_big_fire"], test_t["is_big_fire"]

lgb_temporal = lgb.LGBMClassifier(n_estimators=200, max_depth=8, learning_rate=0.05,
                                   is_unbalance=True, random_state=42, verbose=-1)
lgb_temporal.fit(X_train_t10, y_train_t10)

proba_tlgb10 = lgb_temporal.predict_proba(X_test_t10)[:, 1]
print(f"LightGBM temporal split (v10): ROC-AUC {roc_auc_score(y_test_t10, proba_tlgb10):.3f}")

# threshold tuned on this same split for reference (final versions tune only on calibration years)
from sklearn.metrics import precision_recall_curve
precision_vals, recall_vals, thresholds_pr = precision_recall_curve(y_test_t10, proba_tlgb10)
f1_scores = 2 * (precision_vals * recall_vals) / (precision_vals + recall_vals + 1e-10)
best_threshold_lgb10 = thresholds_pr[np.argmax(f1_scores)]

joblib.dump({"model": lgb_temporal, "feature_cols": feature_cols_v10, "best_threshold": best_threshold_lgb10,
             "label_encoder_province": le_prov}, "model_v10_lightgbm_BEST.pkl")


## v11 — calibration (experimental)

Added Platt and isotonic calibration on top of v10, with reliability checks. Saved separately — did not yet replace v10 as production.

In [ ]:
def logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4)
    return np.log(p / (1 - p))

feature_cols = feature_cols_v10
train = df_model_v10[df_model_v10.YEAR.between(2012, 2021)]
calib = df_model_v10[df_model_v10.YEAR.between(2022, 2024)]

def X_of(df):
    X = df[feature_cols].copy()
    return X.fillna(train[feature_cols].mean())

model = lgb.LGBMClassifier(n_estimators=200, max_depth=8, learning_rate=0.05,
                           is_unbalance=True, random_state=42, verbose=-1)
model.fit(X_of(train), train["is_big_fire"])

raw_ca = model.predict_proba(X_of(calib))[:, 1]
y_ca = calib["is_big_fire"].values
platt = LogisticRegression(C=1e6, max_iter=1000).fit(logit(raw_ca).reshape(-1, 1), y_ca)

from sklearn.isotonic import IsotonicRegression
iso = IsotonicRegression(out_of_bounds="clip").fit(raw_ca, y_ca)

joblib.dump({"model": model, "platt": platt, "iso": iso, "feature_cols": feature_cols,
             "trained_on": "2012-2021", "calibrated_on": "2022-2024"}, "calibrated_model_v11.pkl")


## v12 — first frozen production model

Trained 2012–2021, Platt-calibrated on 2022–2024, alert threshold chosen by best F1 on the **calibration set only**. FWI (Fire Weather Index) was tested separately and dropped — it never earned its place. A known Alberta 2023 satellite data issue was fixed. 2025 was deliberately excluded from training to keep it available as a genuine forward test.

In [ ]:
ths = np.arange(0.05, 0.80, 0.01)
thr = float(ths[int(np.argmax([f1_score(y_ca, platt.predict_proba(logit(raw_ca).reshape(-1, 1))[:, 1] >= t) for t in ths]))])
cal_ca = platt.predict_proba(logit(raw_ca).reshape(-1, 1))[:, 1]
print(f"Alert threshold (chosen on 2022-2024): {thr:.2f}")
print(f"2022-2024: AUC {roc_auc_score(y_ca, cal_ca):.3f} | precision {precision_score(y_ca, cal_ca >= thr):.3f} "
      f"| recall {recall_score(y_ca, cal_ca >= thr):.3f}")

province_map = dict(zip(le_prov.classes_, le_prov.transform(le_prov.classes_)))

joblib.dump({"model": model, "platt": platt, "feature_cols": feature_cols, "threshold": thr,
             "impute_means": train[feature_cols].mean().to_dict(), "province_map": province_map,
             "trained_on": "2012-2021", "calibrated_on": "2022-2024", "threshold_chosen_on": "2022-2024",
             "notes": "no FWI; Alberta 2023 satellite fixed; 2025 excluded"}, "final_model_v12.pkl")
print("Saved final_model_v12.pkl (frozen)")


## Rejected — hyperparameter tuning (Optuna, 80 trials)

Tuned against 2022-2024 as the selection metric. It looked better on that same metric — expected, since it was optimized against it — but that's exactly the kind of result this project treats with suspicion rather than trust. Never adopted; kept for the record as a documented rejection, not silently dropped.

In [ ]:
import optuna

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 800),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "num_leaves": trial.suggest_int("num_leaves", 15, 255),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "is_unbalance": True, "random_state": 42, "verbose": -1,
    }
    m = lgb.LGBMClassifier(**params)
    m.fit(X_of(train), train["is_big_fire"])
    p = m.predict_proba(X_of(calib))[:, 1]
    return average_precision_score(calib["is_big_fire"], p)  # selection metric only — expect optimism

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=80, show_progress_bar=True)

print("Best PR-AUC on 2022-2024 (selection metric — expected to be optimistic):", round(study.best_value, 4))
# Verdict: did not survive a genuine forward-test comparison against v12's untuned defaults. Rejected.


## v13 — road distance

Added `dist_to_road_m` (nearest-road distance, Statistics Canada NRN) — a genuine accessibility signal. Missing values (mostly Parks Canada fires) imputed with the training median. First feature after v12 to show a real, bootstrap-confirmed gain.

In [ ]:
road_dist = pd.read_csv("road_distance_train.csv")
D2 = pd.read_csv("train_table_fixed.csv", parse_dates=["REP_DATE"])
D2 = D2.merge(road_dist, on="NFDBFIREID", how="left")

train_clean = D2[D2.year.between(2012, 2021)]
calib_v13 = D2[D2.year.between(2022, 2024)]

road_median = train_clean["dist_to_road_m"].median()
print(f"Missing road distance in training: {train_clean['dist_to_road_m'].isna().sum()} of {len(train_clean)}")

feature_cols_v13 = feature_cols + ["dist_to_road_m"]
fill_v13 = {c: train_clean[c].mean() for c in ["NDVI", "elevation", "slope"]}
fill_v13["dist_to_road_m"] = road_median

def X_of_v13(df):
    X = df[feature_cols_v13].copy()
    for c, m in fill_v13.items():
        X[c] = X[c].fillna(m)
    return X

model_v13 = lgb.LGBMClassifier(n_estimators=200, max_depth=8, learning_rate=0.05,
                               is_unbalance=True, random_state=42, verbose=-1)
model_v13.fit(X_of_v13(train_clean), train_clean.is_big_fire)

raw_ca = model_v13.predict_proba(X_of_v13(calib_v13))[:, 1]
y_ca = calib_v13["is_big_fire"].values
platt_v13 = LogisticRegression(C=1e6, max_iter=1000).fit(logit(raw_ca).reshape(-1, 1), y_ca)
cal_ca = platt_v13.predict_proba(logit(raw_ca).reshape(-1, 1))[:, 1]

ths = np.arange(0.05, 0.80, 0.01)
thr_v13 = float(ths[int(np.argmax([f1_score(y_ca, cal_ca >= t) for t in ths]))])
print(f"Alert threshold: {thr_v13:.2f} | AUC {roc_auc_score(y_ca, cal_ca):.3f}")

joblib.dump({"model": model_v13, "platt": platt_v13, "feature_cols": feature_cols_v13,
            "threshold": thr_v13, "impute_means": fill_v13, "province_map": province_map,
            "trained_on": "2012-2021", "calibrated_on": "2022-2024",
            "notes": "v12 + distance-to-nearest-road (NRN); 2025 excluded from training"},
           "final_model_v13.pkl")
print("Saved final_model_v13.pkl")


## Rejected — nearby recent-fire count

Tested whether the count of other fires within a radius in the days before a report (a regional fire-outbreak signal) improved on v13. Checked year-by-year across 2022-2024. Not adopted — population density (next) showed a stronger, more consistent signal for the same slot.

In [ ]:
nearby_cols = ["fires_within_25km_7d", "fires_within_50km_7d"]  # built via nearest-neighbor spatial join, omitted here

m_v13_nb = lgb.LGBMClassifier(n_estimators=200, max_depth=8, learning_rate=0.05, is_unbalance=True, random_state=42, verbose=-1)
m_v13_nb.fit(X_of_v13(train_clean), train_clean.is_big_fire)

m_nearby = lgb.LGBMClassifier(n_estimators=200, max_depth=8, learning_rate=0.05, is_unbalance=True, random_state=42, verbose=-1)
m_nearby.fit(D2[D2.year.between(2012, 2021)][feature_cols_v13 + nearby_cols], train_clean.is_big_fire)

for yr in [2022, 2023, 2024]:
    df_yr = D2[D2.year == yr]
    y_yr = df_yr.is_big_fire
    p_r = m_v13_nb.predict_proba(X_of_v13(df_yr))[:, 1]
    p_n = m_nearby.predict_proba(df_yr[feature_cols_v13 + nearby_cols])[:, 1]
    print(f"{yr}: road-only PR {average_precision_score(y_yr, p_r):.4f} | +nearby PR {average_precision_score(y_yr, p_n):.4f}")
# Verdict: no consistent gain. Rejected.


## v14 — population density

Added `pop_within_10km` and `pop_within_25km` (Kontur Population Dataset). Adopted on evidence across 4 of 5 tested periods — 2025 showed a null result rather than a negative one, which is the standard this project holds a change to: no confirmed harm, real gain elsewhere.

In [ ]:
pop_cols = ["pop_within_10km", "pop_within_25km"]
feature_cols_v14 = feature_cols_v13 + pop_cols

train_14 = D2[D2.year.between(2012, 2021)]
calib_14 = D2[D2.year.between(2022, 2024)]

fill_14 = {c: train_14[c].mean() for c in ["NDVI", "elevation", "slope", "dist_to_road_m"]}
fill_14.update({c: 0 for c in pop_cols})

def X_of_14(df):
    X = df[feature_cols_v14].copy()
    for c, m in fill_14.items():
        X[c] = X[c].fillna(m)
    return X

model_v14 = lgb.LGBMClassifier(n_estimators=200, max_depth=8, learning_rate=0.05, is_unbalance=True, random_state=42, verbose=-1)
model_v14.fit(X_of_14(train_14), train_14.is_big_fire)

raw_ca = model_v14.predict_proba(X_of_14(calib_14))[:, 1]
y_ca = calib_14["is_big_fire"].values
platt_v14 = LogisticRegression(C=1e6, max_iter=1000).fit(logit(raw_ca).reshape(-1, 1), y_ca)
cal_ca = platt_v14.predict_proba(logit(raw_ca).reshape(-1, 1))[:, 1]

ths = np.arange(0.05, 0.80, 0.01)
thr_v14 = float(ths[int(np.argmax([f1_score(y_ca, cal_ca >= t) for t in ths]))])
print(f"Alert threshold: {thr_v14:.2f} | AUC {roc_auc_score(y_ca, cal_ca):.3f}")

joblib.dump({"model": model_v14, "platt": platt_v14, "feature_cols": feature_cols_v14, "threshold": thr_v14,
            "impute_means": fill_14, "province_map": province_map,
            "trained_on": "2012-2021", "calibrated_on": "2022-2024",
            "notes": "v13 + population density (Kontur); adopted on 4-of-5-year evidence, 2025 null"},
           "final_model_v14.pkl")
print("Saved final_model_v14.pkl")


## v14.1 — NT/YT slope data-quality fix

While investigating consistently weaker NT/YT performance, found all 3,016 NT/YT training fires had wrong slope values (near-zero) — the terrain fetch's Copernicus fallback for above-60°N locations was implemented for elevation but never wired up for slope. Fixed, retrained, and validated with bootstrap confidence intervals across every province and both forward-test years before adoption.

In [ ]:
slope_fix = pd.read_csv("nt_yt_slope_fix.csv")  # fresh Earth Engine query, correct values

train_patched = D2.merge(slope_fix, on="NFDBFIREID", how="left")
train_patched["slope"] = train_patched["slope_fixed"].where(train_patched["slope_fixed"].notna(), train_patched["slope"])
train_patched = train_patched.drop(columns=["slope_fixed"])

train_141 = train_patched[train_patched.year.between(2012, 2021)]
calib_141 = train_patched[train_patched.year.between(2022, 2024)]

fill_141 = {c: train_141[c].mean() for c in ["NDVI", "elevation", "slope", "dist_to_road_m"]}
fill_141.update({c: 0 for c in pop_cols})

def X_of_141(df):
    X = df[feature_cols_v14].copy()
    for c, m in fill_141.items():
        X[c] = X[c].fillna(m)
    return X

model_v141 = lgb.LGBMClassifier(n_estimators=200, max_depth=8, learning_rate=0.05, is_unbalance=True, random_state=42, verbose=-1)
model_v141.fit(X_of_141(train_141), train_141.is_big_fire)

raw_ca = model_v141.predict_proba(X_of_141(calib_141))[:, 1]
y_ca = calib_141["is_big_fire"].values
platt_141 = LogisticRegression(C=1e6, max_iter=1000).fit(logit(raw_ca).reshape(-1, 1), y_ca)

bundle_v14_1 = {
    "model": model_v141, "platt_calibrator": platt_141, "feature_cols": feature_cols_v14,
    "fill_means": fill_141, "version": "v14.1",
    "notes": "Bug fix: corrected NT/YT slope (Copernicus fallback was missing for SRTM polar gap, ~60N+). "
             "No feature changes, no new data added.",
}
joblib.dump(bundle_v14_1, "final_model_v14_1.pkl")
print("Saved final_model_v14_1.pkl — current production model")


## Validation — bootstrap confidence intervals

Every adopted change in this project is gated by this check: a real effect should clear zero on *every* tested period, not just look better on a point estimate.

In [ ]:
def bootstrap_delta(y, p1, p2, n_boot=2000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(y)
    deltas = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        deltas[i] = average_precision_score(y[idx], p2[idx]) - average_precision_score(y[idx], p1[idx])
    lo, hi = np.percentile(deltas, [2.5, 97.5])
    return lo, hi, (deltas > 0).mean()

# Example: v14 (pre-fix) vs v14.1 (post-fix) on a forward-test year
# lo, hi, frac = bootstrap_delta(y_2026, p_v14, p_v14_1)
# print(f"delta CI [{lo:+.4f}, {hi:+.4f}], {frac*100:.1f}% favor v14.1")
